# EOQ Lab — Public Education Data Collection

**States:** Hawaii, Virginia  
**Goal:** Download publicly available education files (CSV, XLSX, PDF, etc.) and document them.

Run this notebook **from top to bottom**. Each section builds on the previous one.

---
## How to use this notebook
1. Run the install cell once (first time only).
2. Run setup cells to create folders.
3. Run Phase 1 to download a few real files and prove the pipeline works.
4. Later sections (Phases 2–6) will be added step by step.

---
## Step 0.1 — Install required packages

This downloads Python libraries into **the same environment** your notebook kernel uses.

After it finishes, if imports fail, use **Kernel → Restart** and run cells again.

In [1]:
# %pip installs into the active Jupyter kernel (preferred over !pip)
%pip install -q requests beautifulsoup4 pandas openpyxl pyyaml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\saihaj\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


---
## Step 0.2 — Imports and project paths

We define where files will be saved. All paths are relative to the **project root** (the folder that contains `data/` and `notebooks/`).

In [1]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import unquote, urlparse

import pandas as pd
import requests

# --- Find project root ---
# If you run the notebook from notebooks/, go up one level.
# If you run from project root, stay there.
cwd = Path.cwd()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "notebooks").is_dir() and (cwd / "data").is_dir():
    PROJECT_ROOT = cwd
else:
    # Fallback: assume parent of notebooks folder
    PROJECT_ROOT = cwd.parent if (cwd.parent / "data").exists() else cwd

print("Project root:", PROJECT_ROOT.resolve())

# --- Folder layout ---
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_CLEANED = PROJECT_ROOT / "data" / "cleaned"
LOGS_DIR = PROJECT_ROOT / "logs"
DOCS_DIR = PROJECT_ROOT / "docs"

# Subcategories we use for organizing downloads
CATEGORIES = [
    "test_scores",
    "enrollment",
    "financials",
    "teachers",
    "discipline",
    "other",
]

STATES = ["hawaii", "virginia"]

# HTTP header: many government sites block anonymous/bot traffic.
# This identifies our script politely (still public data only).
DEFAULT_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (compatible; EOQ-Lab-Education-Data-Collection/1.0; "
        "academic research; public data)"
    )
}

Project root: C:\Users\saihaj\Documents\Data Collection - EOQ Lab


---
## Step 0.3 — Create folders on disk

We create `data/raw/{state}/{category}/` for each state and category.
**Rule:** Never edit files in `data/raw/` by hand after download — they are your proof of what the state published.

In [2]:
for folder in [DATA_RAW, DATA_CLEANED, LOGS_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

for state in STATES:
    for category in CATEGORIES:
        (DATA_RAW / state / category).mkdir(parents=True, exist_ok=True)

(DATA_RAW / "federal").mkdir(parents=True, exist_ok=True)

print("Folders ready under:", DATA_RAW.resolve())

Folders ready under: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw


---
## Step 0.4 — Download helper function

This is the function we will reuse for **every** file in later phases.

**What it does:**
1. Sends an HTTP GET request to the file URL (like clicking Download).
2. Saves the bytes to `data/raw/{state}/{category}/`.
3. Appends one line to `logs/download_log.jsonl` (audit trail).

**Parameters:**
- `url` — direct link to `.xlsx`, `.csv`, `.pdf`, etc.
- `state` — `"hawaii"` or `"virginia"`
- `category` — folder name, e.g. `"enrollment"`
- `description` — short text for your supervisor's catalog

In [3]:
DOWNLOAD_LOG = LOGS_DIR / "download_log.jsonl"
MANIFEST_CSV = LOGS_DIR / "manifest.csv"


def _filename_from_url(url: str, response: requests.Response) -> str:
    """Pick a safe local filename from URL or Content-Disposition header."""
    cd = response.headers.get("Content-Disposition", "")
    match = re.search(r'filename="?([^";]+)"?', cd)
    if match:
        return match.group(1).strip()
    path_name = Path(unquote(urlparse(url).path)).name
    return path_name or "download.bin"


def download_file(
    url: str,
    state: str,
    category: str,
    description: str,
    *,
    skip_if_exists: bool = True,
    timeout_seconds: int = 120,
) -> dict:
    """
    Download one public file and return a metadata dict (also logged to JSONL).
    """
    state = state.lower()
    if state not in STATES:
        raise ValueError(f"state must be one of {STATES}, got {state!r}")
    if category not in CATEGORIES:
        raise ValueError(f"category must be one of {CATEGORIES}, got {category!r}")

    dest_dir = DATA_RAW / state / category
    dest_dir.mkdir(parents=True, exist_ok=True)

    # We do not know the filename until we GET (or read headers).
    # First request: stream so large files do not fill RAM.
    with requests.get(
        url,
        headers=DEFAULT_HEADERS,
        timeout=timeout_seconds,
        stream=True,
    ) as response:
        response.raise_for_status()  # fail clearly on 404, 403, etc.
        filename = _filename_from_url(url, response)
        dest_path = dest_dir / filename

        if skip_if_exists and dest_path.exists():
            record = {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "status": "skipped_exists",
                "state": state,
                "category": category,
                "description": description,
                "url": url,
                "local_path": str(dest_path.relative_to(PROJECT_ROOT)),
                "bytes": dest_path.stat().st_size,
            }
            with DOWNLOAD_LOG.open("a", encoding="utf-8") as log_file:
                log_file.write(json.dumps(record) + "\n")
            return record

        # Write file in chunks (8 KB at a time)
        bytes_written = 0
        with dest_path.open("wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
                    bytes_written += len(chunk)

    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "status": "downloaded",
        "state": state,
        "category": category,
        "description": description,
        "url": url,
        "local_path": str(dest_path.relative_to(PROJECT_ROOT)),
        "bytes": bytes_written,
    }
    with DOWNLOAD_LOG.open("a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(record) + "\n")
    return record


def rebuild_manifest_from_log() -> pd.DataFrame:
    """Read download_log.jsonl and write logs/manifest.csv (handy for your supervisor)."""
    if not DOWNLOAD_LOG.exists():
        return pd.DataFrame()
    rows = []
    with DOWNLOAD_LOG.open(encoding="utf-8") as log_file:
        for line in log_file:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    df.to_csv(MANIFEST_CSV, index=False)
    return df


print("Helper ready: download_file(...)")

Helper ready: download_file(...)


---
# Phase 1 — Direct downloads (proof of concept)

These URLs point **directly to files** (not HTML menu pages).
Python does the same thing as you clicking **Download** in the browser.

Official sources:
- **Hawaii:** HIDOE School Reports (enrollment Excel, Title I PDF)
- **Virginia:** Virginia Open Data Portal (SOL / test results file)

In [4]:
# Each item is one file to download.
# We will build longer lists automatically in Phase 2 (VA API) and Phase 3 (BeautifulSoup).
PHASE1_DOWNLOADS = [
  {
    "state": "hawaii",
    "category": "enrollment",
    "description": "HIDOE official enrollment counts 2025-26",
    "url": "https://hawaiipublicschools.org/wp-content/uploads/HIDOEenrollment_2025-26.xlsx",
  },
  {
    "state": "hawaii",
    "category": "other",
    "description": "HIDOE Title I schools list 2025-26 (PDF)",
    "url": "https://hawaiipublicschools.org/wp-content/uploads/TitleI-2025-26-revised-122025.pdf",
  },
  {
    "state": "virginia",
    "category": "test_scores",
    "description": "Virginia SOL state pass rates by test 2017-2023 (Open Data)",
    "url": (
      "https://data.virginia.gov/dataset/5002d90c-8f8a-45b8-b72f-7ccf99b85438/"
      "resource/eed1d4b3-633e-496e-820b-b13115b81e4f/download/"
      "state_pass_rates_by_test_2017-2023.xlsx"
    ),
  },
]

pd.DataFrame(PHASE1_DOWNLOADS)

,state,category,description,url
0,hawaii,enrollment,HIDOE official enrollment counts 2025-26,https://hawaiipublicschools.org/wp-content/upl...
1,hawaii,other,HIDOE Title I schools list 2025-26 (PDF),https://hawaiipublicschools.org/wp-content/upl...
2,virginia,test_scores,Virginia SOL state pass rates by test 2017-202...,https://data.virginia.gov/dataset/5002d90c-8f8...


In [5]:
# Loop: download each file and collect results
phase1_results = []

for item in PHASE1_DOWNLOADS:
    print(f"Downloading: {item['description']}")
    result = download_file(
        url=item["url"],
        state=item["state"],
        category=item["category"],
        description=item["description"],
    )
    phase1_results.append(result)
    print(f"  -> {result['status']}: {result['local_path']} ({result['bytes']:,} bytes)\n")

manifest_df = rebuild_manifest_from_log()
manifest_df

Downloading: HIDOE official enrollment counts 2025-26
  -> skipped_exists: data\raw\hawaii\enrollment\HIDOEenrollment_2025-26.xlsx (41,551 bytes)

Downloading: HIDOE Title I schools list 2025-26 (PDF)
  -> skipped_exists: data\raw\hawaii\other\TitleI-2025-26-revised-122025.pdf (69,643 bytes)

Downloading: Virginia SOL state pass rates by test 2017-2023 (Open Data)
  -> skipped_exists: data\raw\virginia\test_scores\state_pass_rates_by_test_2017-2023.xlsx (13,280 bytes)



,timestamp_utc,status,state,category,description,url,local_path,bytes
0,2026-05-23T21:55:42.011861+00:00,skipped_exists,hawaii,enrollment,HIDOE official enrollment counts 2025-26,https://hawaiipublicschools.org/wp-content/upl...,data\raw\hawaii\enrollment\HIDOEenrollment_202...,41551
1,2026-05-23T21:55:42.412875+00:00,downloaded,hawaii,other,HIDOE Title I schools list 2025-26 (PDF),https://hawaiipublicschools.org/wp-content/upl...,data\raw\hawaii\other\TitleI-2025-26-revised-1...,69643
2,2026-05-23T21:55:43.434676+00:00,skipped_exists,virginia,test_scores,Virginia SOL state pass rates by test 2017-202...,https://data.virginia.gov/dataset/5002d90c-8f8...,data\raw\virginia\test_scores\state_pass_rates...,13280
3,2026-05-23T23:09:08.432225+00:00,skipped_exists,hawaii,enrollment,HIDOE official enrollment counts 2025-26,https://hawaiipublicschools.org/wp-content/upl...,data\raw\hawaii\enrollment\HIDOEenrollment_202...,41551
4,2026-05-23T23:09:08.707050+00:00,skipped_exists,hawaii,other,HIDOE Title I schools list 2025-26 (PDF),https://hawaiipublicschools.org/wp-content/upl...,data\raw\hawaii\other\TitleI-2025-26-revised-1...,69643
5,2026-05-23T23:09:09.636168+00:00,skipped_exists,virginia,test_scores,Virginia SOL state pass rates by test 2017-202...,https://data.virginia.gov/dataset/5002d90c-8f8...,data\raw\virginia\test_scores\state_pass_rates...,13280


---
## Phase 1 — Check your files

Open the folders below in File Explorer. You should see the downloaded files.

Also open `logs/manifest.csv` — this is the table I'll share with my supervisor.

In [6]:
# List all files currently under data/raw/ (recursive)
raw_files = sorted(DATA_RAW.rglob("*"))
raw_files = [p for p in raw_files if p.is_file()]

for path in raw_files:
    rel = path.relative_to(PROJECT_ROOT)
    print(f"{rel}  ({path.stat().st_size:,} bytes)")

print(f"\nTotal files in data/raw: {len(raw_files)}")

data\raw\hawaii\enrollment\HIDOEenrollment_2025-26.xlsx  (41,551 bytes)
data\raw\hawaii\other\TitleI-2025-26-revised-122025.pdf  (69,643 bytes)
data\raw\virginia\test_scores\state_pass_rates_by_test_2017-2023.xlsx  (13,280 bytes)

Total files in data/raw: 3


---
# Phase 2 — Virginia Open Data API (CKAN)

**Goal:** Automatically discover **all** Department of Education datasets on [data.virginia.gov](https://data.virginia.gov) and download their files.

**How this differs from Phase 1:**
- Phase 1 used **hand-picked file URLs** (we knew the link already).
- Phase 2 asks Virginia's **API** (a machine-readable catalog): *"List every dataset from the Department of Education and give me download links."*

**What the API returns:** JSON (text describing datasets). We read that JSON, extract real file URLs (`.xlsx`, `.csv`, `.pdf`), then reuse `download_file()` from Phase 1.

**Run the Phase 2 cells in order** (after Phase 0 and Phase 1 have already been run in this session).

## Phase 2.1 — API settings and helper functions

Virginia uses **CKAN**, a common open-data platform. Its API base URL is:

`https://data.virginia.gov/api/3/action/`

We call **`package_search`** with a filter for the **Department of Education** organization. That returns a list of datasets; each dataset has one or more **resources** (actual download links).

In [7]:
import time

# --- Virginia Open Data (CKAN) API base URL ---
VA_CKAN_API = "https://data.virginia.gov/api/3/action"

# Search filter: only datasets published by Virginia Department of Education
VA_ORG_QUERY = "organization:department-of-education"

# File types we know how to save as downloadable files.
# We skip "LINK" because those are often external web pages, not direct files.
ALLOWED_FORMATS = {"CSV", "XLSX", "XLS", "PDF", "ZIP"}

# Where we save the discovery table (before downloading)
VA_DISCOVERED_CSV = LOGS_DIR / "va_discovered_links.csv"

# Pause between downloads (seconds) — polite to the server when downloading many files
SECONDS_BETWEEN_DOWNLOADS = 0.5


def ckan_package_search(query: str, rows: int = 100, start: int = 0) -> dict:
    """
    Call CKAN 'package_search' and return the JSON 'result' object.

    Parameters:
      query  — CKAN search string (like a search box on the website)
      rows   — how many datasets to return per page (max 100)
      start  — offset for pagination (0, 100, 200, ...)

    Returns:
      dict with keys like 'count' (total datasets) and 'results' (list of datasets)
    """
    url = f"{VA_CKAN_API}/package_search"
    params = {"q": query, "rows": rows, "start": start}

    response = requests.get(url, params=params, headers=DEFAULT_HEADERS, timeout=60)
    response.raise_for_status()

    payload = response.json()
    if not payload.get("success"):
        raise RuntimeError(f"CKAN API error: {payload}")

    return payload["result"]


def guess_category(title: str, description: str = "") -> str:
    """
    Guess which project folder to use based on words in the dataset title.

    This is a simple keyword match — not perfect, but good enough for sorting files.
    """
    text = f"{title} {description}".lower()

    rules = [
        ("test_scores", ["sol", "assessment", "test", "score", "pass rate", "accreditation"]),
        ("enrollment", ["enrollment", "membership", "fall membership", "student count", "adm"]),
        ("financials", ["finance", "expenditure", "per pupil", "budget", "funding", "salary"]),
        ("teachers", ["teacher", "staff", "personnel", "licens", "qualification"]),
        ("discipline", ["discipline", "suspension", "expulsion", "absenteeism", "climate", "dropout"]),
    ]

    for category, keywords in rules:
        if any(word in text for word in keywords):
            return category

    return "other"


print("Phase 2 helpers loaded.")

Phase 2 helpers loaded.


## Phase 2.2 — Discover all VA file links via the API

This cell:
1. Pages through the API (100 datasets at a time until we have all ~244).
2. For each dataset, loops through its **resources** (download links).
3. Keeps only resources with formats we want (CSV, XLSX, PDF, etc.).
4. Builds a pandas table and saves `logs/va_discovered_links.csv`.

**You are not downloading yet** — only building the shopping list.

In [8]:
# --- Step 1: fetch ALL Department of Education datasets (paginated) ---
PAGE_SIZE = 100
start = 0
all_packages = []

while True:
  page = ckan_package_search(VA_ORG_QUERY, rows=PAGE_SIZE, start=start)
  batch = page["results"]
  all_packages.extend(batch)

  total = page["count"]
  print(f"Fetched datasets {start + 1}–{start + len(batch)} of {total}")

  start += PAGE_SIZE
  if start >= total:
    break

print(f"\nTotal datasets retrieved: {len(all_packages)}")

# --- Step 2: flatten datasets → one row per downloadable file ---
discovered_rows = []

for package in all_packages:
  title = package.get("title") or ""
  notes = package.get("notes") or ""  # dataset description (may contain HTML)
  category = guess_category(title, notes)

  for resource in package.get("resources", []):
    fmt = (resource.get("format") or "").upper().strip()
    url = (resource.get("url") or "").strip()

    # Skip empty URLs or formats we don't handle (especially LINK = web page)
    if not url or fmt not in ALLOWED_FORMATS:
      continue

    discovered_rows.append(
      {
        "state": "virginia",
        "category": category,
        "dataset_title": title,
        "resource_name": resource.get("name") or "",
        "format": fmt,
        "url": url,
        "description": f"{title} — {resource.get('name') or fmt}",
      }
    )

# --- Step 3: build table, remove duplicate URLs, save CSV ---
va_links_df = pd.DataFrame(discovered_rows)

if not va_links_df.empty:
  before = len(va_links_df)
  va_links_df = va_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
  print(f"Downloadable file links: {len(va_links_df)} (removed {before - len(va_links_df)} duplicate URLs)")
  va_links_df.to_csv(VA_DISCOVERED_CSV, index=False)
  print(f"Saved discovery table → {VA_DISCOVERED_CSV.relative_to(PROJECT_ROOT)}")
else:
  print("No links found — check internet connection or API status.")

# Preview first 10 rows
va_links_df.head(10)

Fetched datasets 1–100 of 244
Fetched datasets 101–200 of 244
Fetched datasets 201–244 of 244

Total datasets retrieved: 244
Downloadable file links: 229 (removed 0 duplicate URLs)
Saved discovery table → logs\va_discovered_links.csv


,state,category,dataset_title,resource_name,format,url,description
0,virginia,other,Local and Regional Schools and Centers 2021 - ...,2021-22-local-and-regional-schools,XLSX,https://data.virginia.gov/dataset/89a641ec-9df...,Local and Regional Schools and Centers 2021 - ...
1,virginia,test_scores,Cohort Graduation and Dropout Report 2020,Cohort Graduation and Dropout Rate 2020,CSV,https://data.virginia.gov/dataset/c13dbcbb-92f...,Cohort Graduation and Dropout Report 2020 — Co...
2,virginia,enrollment,2003-2004 Fall Membership School Totals by Grade,2003-2004 Fall Membership School Totals by Grade,XLSX,https://data.virginia.gov/dataset/21daa5fe-730...,2003-2004 Fall Membership School Totals by Gra...
3,virginia,other,Home Schooled Students & Religious Exemptions ...,Home Schooled Students & Religious Exemptions ...,XLS,https://data.virginia.gov/dataset/320b9209-3a7...,Home Schooled Students & Religious Exemptions ...
4,virginia,other,Graduates & Completers - Graduates by Continui...,2019-2020-division-totals-continuing-ed,XLS,https://data.virginia.gov/dataset/c1f08fe5-62b...,Graduates & Completers - Graduates by Continui...
5,virginia,other,2002-2003 Gifted Education Annual Total Report,2003Gifted Education Annual Total Report,PDF,https://data.virginia.gov/dataset/8e0e3df2-000...,2002-2003 Gifted Education Annual Total Report...
6,virginia,enrollment,"Administrative, Service and Support Personnel ...",Fiscal Year 2021,XLSX,https://data.virginia.gov/dataset/311ecec4-029...,"Administrative, Service and Support Personnel ..."
7,virginia,enrollment,2000-2001 Fall Membership School Totals by Grade,2000-2001 Fall Membership School Totals by Grade,XLSX,https://data.virginia.gov/dataset/c08429df-b62...,2000-2001 Fall Membership School Totals by Gra...
8,virginia,enrollment,2000-2001 Gifted Annual Report by Membership,2000-2001 Gifted Annual Report by Membership,PDF,https://data.virginia.gov/dataset/8dbf75de-cf6...,2000-2001 Gifted Annual Report by Membership —...
9,virginia,other,Local and Regional Schools and Centers 2023 - ...,2023-24 Local and Regional Schools,XLSX,https://data.virginia.gov/dataset/137bf667-69e...,Local and Regional Schools and Centers 2023 - ...


## Phase 2.3 — Download every discovered Virginia file

This reuses **`download_file()`** from Step 0.4 — same function as Phase 1.

- Each row in `va_links_df` → one call to `download_file(...)`.
- Files land in `data/raw/virginia/{category}/`.
- Already-downloaded files (e.g. the SOL file from Phase 1) log as `skipped_exists`.
- This may take several minutes (~200+ files). That is normal.

In [9]:
# --- Download loop: one file per row in the discovery table ---
phase2_results = []
failures = []

for i, row in va_links_df.iterrows():
  print(f"[{i + 1}/{len(va_links_df)}] {row['description'][:70]}...")

  try:
    result = download_file(
      url=row["url"],
      state=row["state"],
      category=row["category"],
      description=row["description"],
    )
    phase2_results.append(result)
    print(f"  -> {result['status']}: {result['local_path']}")

  except requests.HTTPError as err:
    # Log failed downloads but keep going (don't stop the whole batch)
    failures.append({"url": row["url"], "error": str(err)})
    print(f"  -> FAILED: {err}")

  # Small pause so we don't hammer the server
  time.sleep(SECONDS_BETWEEN_DOWNLOADS)

# --- Summary ---
manifest_df = rebuild_manifest_from_log()

print("\n--- Phase 2 summary ---")
print(f"Attempted: {len(va_links_df)}")
print(f"Logged results: {len(phase2_results)}")
print(f"Failed: {len(failures)}")

if failures:
  pd.DataFrame(failures).head()

[1/229] Local and Regional Schools and Centers 2021 - 2022 — 2021-22-local-and...
  -> downloaded: data\raw\virginia\other\2021-22-local-and-regional-schools-3-1.xlsx
[2/229] Cohort Graduation and Dropout Report 2020 — Cohort Graduation and Drop...
  -> downloaded: data\raw\virginia\test_scores\cohort_statistics-7.csv
[3/229] 2003-2004 Fall Membership School Totals by Grade — 2003-2004 Fall Memb...
  -> downloaded: data\raw\virginia\enrollment\03-stg.xls
[4/229] Home Schooled Students & Religious Exemptions 2015-16 — Home Schooled ...
  -> downloaded: data\raw\virginia\other\2015_2016.xlsx
[5/229] Graduates & Completers - Graduates by Continuing Education Plans 2019 ...
  -> downloaded: data\raw\virginia\other\2019-2020-division-totals-continuing-ed-1.xls
[6/229] 2002-2003 Gifted Education Annual Total Report — 2003Gifted Education ...
  -> downloaded: data\raw\virginia\other\2003_gifted.pdf
[7/229] Administrative, Service and Support Personnel Positions by Function - ...
  -> download

In [10]:
# --- Count Virginia files by category after Phase 2 ---
va_files = [p for p in (DATA_RAW / "virginia").rglob("*") if p.is_file()]

counts = {}
for path in va_files:
  # path looks like: data/raw/virginia/enrollment/file.xlsx → category is parent folder name
  category = path.parent.name
  counts[category] = counts.get(category, 0) + 1

summary = pd.DataFrame(
  [{"category": k, "file_count": v} for k, v in sorted(counts.items())]
)

print(f"Total Virginia files in data/raw/virginia/: {len(va_files)}")
summary

Total Virginia files in data/raw/virginia/: 220


,category,file_count
0,discipline,6
1,enrollment,79
2,financials,4
3,other,107
4,test_scores,24


---
# Phase 3 — BeautifulSoup link harvest (HTML pages)

**Goal:** Automatically find download links on **HTML report pages** — especially **Hawaii**, which has no big open-data API like Virginia.

**How this differs from Phase 2:**
- Phase 2 asked an **API** for a JSON catalog (Virginia).
- Phase 3 fetches a **webpage** (HTML), parses it with **BeautifulSoup**, finds every `<a href="...xlsx">` link, then downloads with `download_file()`.

**Virginia note:** `doe.virginia.gov` often blocks automated access (403). Phase 2 already collected ~220 VA files from the Open Data portal, so Phase 3 focuses on **Hawaii**.

**Run Phase 3 cells in order** (after Steps 0.2, 0.4, and Phase 2.1 helpers like `guess_category` are loaded).

## Phase 3.1 — Settings and helper functions for HTML harvesting

**BeautifulSoup** reads HTML and lets us search for tags like `<a href="...">`.

We will:
1. Download the HTML of a seed page with `requests`.
2. Parse it with BeautifulSoup.
3. Collect links whose URLs end in `.xlsx`, `.csv`, `.pdf`, etc.
4. Build a table (like Phase 2) and save `logs/hi_discovered_links.csv`.

In [11]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

# --- Output file for Hawaii links found on HTML pages ---
HI_DISCOVERED_CSV = LOGS_DIR / "hi_discovered_links.csv"

# File extensions we treat as downloadable data files
FILE_EXTENSIONS = (".xlsx", ".xls", ".csv", ".pdf", ".zip")

# Slightly more browser-like headers for HTML pages (some sites block bare bots)
HTML_HEADERS = {
    **DEFAULT_HEADERS,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

# --- Seed pages: official HIDOE index pages that list many file links ---
# We only scan pages we trust (state education domain).
HARVEST_SEED_PAGES = [
    {
        "state": "hawaii",
        "label": "HIDOE School Reports (enrollment, Title I, etc.)",
        "url": "https://hawaiipublicschools.org/data-reports/school-reports/",
    },
    {
        "state": "hawaii",
        "label": "Strive HI Dashboard page",
        "url": "https://hawaiipublicschools.org/about/organization/strive-hi-dashboard/",
    },
]


def fetch_page_html(page_url: str) -> str:
    """
    Download the HTML text of a webpage.

    Returns the page source as a string (what you see with View Page Source).
    """
    response = requests.get(page_url, headers=HTML_HEADERS, timeout=60)
    response.raise_for_status()
    return response.text


def extract_file_links(page_url: str, html: str, state: str) -> list[dict]:
    """
    Parse HTML and return a list of dicts — one row per downloadable file link.

    Looks for <a href="..."> where href ends with .xlsx, .pdf, etc.
    """
    soup = BeautifulSoup(html, "html.parser")
    rows = []

    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()

        # Convert relative links like "/DOE Forms/file.xlsx" into full URLs
        full_url = urljoin(page_url, href)

        # Only keep links that look like data files (by extension in the URL path)
        path_lower = urlparse(full_url).path.lower()
        if not path_lower.endswith(FILE_EXTENSIONS):
            continue

        # Text visible on the page for this link (e.g. "Enrollment Data (2025-26)")
        link_text = tag.get_text(" ", strip=True)

        # Reuse keyword rules from Phase 2 to pick a folder category
        category = guess_category(link_text, page_url)

        rows.append(
            {
                "state": state,
                "category": category,
                "link_text": link_text,
                "url": full_url,
                "source_page": page_url,
                "description": f"{link_text} (from {page_url})",
            }
        )

    return rows


print("Phase 3 helpers loaded.")
print(f"Seed pages to scan: {len(HARVEST_SEED_PAGES)}")

Phase 3 helpers loaded.
Seed pages to scan: 2


## Phase 3.2 — Scan seed pages and build the Hawaii link table

This cell loops over each seed page, extracts file links, removes duplicates, and saves the CSV.

**No file downloads yet** — discovery only (same pattern as Phase 2.2).

In [12]:
all_html_rows = []

for seed in HARVEST_SEED_PAGES:
    page_url = seed["url"]
    state = seed["state"]
    label = seed["label"]

    print(f"Scanning: {label}")
    print(f"  URL: {page_url}")

    try:
        html = fetch_page_html(page_url)
        links = extract_file_links(page_url, html, state)
        print(f"  Found {len(links)} file link(s) on this page\n")
        all_html_rows.extend(links)

    except requests.HTTPError as err:
        # If a page blocks us (403), log and continue with other pages
        print(f"  SKIPPED (HTTP error): {err}\n")

# --- Build pandas table ---
hi_links_df = pd.DataFrame(all_html_rows)

if not hi_links_df.empty:
    before = len(hi_links_df)
    # Same URL might appear twice on a page — keep one row per URL
    hi_links_df = hi_links_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    print(f"Total unique Hawaii file links: {len(hi_links_df)} (removed {before - len(hi_links_df)} duplicate URLs)")

    hi_links_df.to_csv(HI_DISCOVERED_CSV, index=False)
    print(f"Saved → {HI_DISCOVERED_CSV.relative_to(PROJECT_ROOT)}")
else:
    print("No links found. Check internet connection or seed URLs.")

hi_links_df

Scanning: HIDOE School Reports (enrollment, Title I, etc.)
  URL: https://hawaiipublicschools.org/data-reports/school-reports/
  Found 17 file link(s) on this page

Scanning: Strive HI Dashboard page
  URL: https://hawaiipublicschools.org/about/organization/strive-hi-dashboard/
  Found 1 file link(s) on this page

Total unique Hawaii file links: 17 (removed 1 duplicate URLs)
Saved → logs\hi_discovered_links.csv


,state,category,link_text,url,source_page,description
0,hawaii,enrollment,Enrollment Data (2025-26) (Excel),https://hawaiipublicschools.org/wp-content/upl...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2025-26) (Excel) (from https:...
1,hawaii,enrollment,Enrollment Data (2024-25) (Excel),https://hawaiipublicschools.org/wp-content/upl...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2024-25) (Excel) (from https:...
2,hawaii,enrollment,Enrollment Data (2023-24) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2023-24) (Excel) (from https:...
3,hawaii,enrollment,Enrollment Data (2022-23) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2022-23) (Excel) (from https:...
4,hawaii,enrollment,Enrollment Data (2021-22) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2021-22) (Excel) (from https:...
5,hawaii,enrollment,Enrollment Data (2020-21) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2020-21) (Excel) (from https:...
6,hawaii,enrollment,Enrollment Data (2019-20) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2019-20) (Excel) (from https:...
7,hawaii,enrollment,Enrollment Data (2018-19) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2018-19) (Excel) (from https:...
8,hawaii,enrollment,Enrollment Data (2017-18) (Excel),https://hawaiipublicschools.org/DOE%20Forms/En...,https://hawaiipublicschools.org/data-reports/s...,Enrollment Data (2017-18) (Excel) (from https:...
9,hawaii,other,"Title I Schools in Hawaiʻi, SY 2026-27",https://hawaiipublicschools.org/wp-content/upl...,https://hawaiipublicschools.org/data-reports/s...,"Title I Schools in Hawaiʻi, SY 2026-27 (from h..."


## Phase 3.3 — Download every discovered Hawaii file

Same **`download_file()`** function as Phases 1 and 2.

- Files go to `data/raw/hawaii/{category}/`.
- Links you already downloaded in Phase 1 (enrollment 2025-26, Title I PDF) will log as **`skipped_exists`** — that is normal.

In [13]:
phase3_results = []
phase3_failures = []

for i, row in hi_links_df.iterrows():
    print(f"[{i + 1}/{len(hi_links_df)}] {row['link_text'][:70]}")

    try:
        result = download_file(
            url=row["url"],
            state=row["state"],
            category=row["category"],
            description=row["description"],
        )
        phase3_results.append(result)
        print(f"  -> {result['status']}: {result['local_path']}")

    except requests.HTTPError as err:
        phase3_failures.append({"url": row["url"], "error": str(err)})
        print(f"  -> FAILED: {err}")

    # Polite pause between requests
    time.sleep(SECONDS_BETWEEN_DOWNLOADS)

manifest_df = rebuild_manifest_from_log()

print("\n--- Phase 3 summary ---")
print(f"Links discovered: {len(hi_links_df)}")
print(f"Download attempts logged: {len(phase3_results)}")
print(f"Failed: {len(phase3_failures)}")

if phase3_failures:
    pd.DataFrame(phase3_failures)

[1/17] Enrollment Data (2025-26) (Excel)
  -> skipped_exists: data\raw\hawaii\enrollment\HIDOEenrollment_2025-26.xlsx
[2/17] Enrollment Data (2024-25) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\HIDOEenrollment_2024-25.xlsx
[3/17] Enrollment Data (2023-24) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2023-24.xlsx
[4/17] Enrollment Data (2022-23) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2022-23.xlsx
[5/17] Enrollment Data (2021-22) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2021-22.xlsx
[6/17] Enrollment Data (2020-21) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2020-21.xlsx
[7/17] Enrollment Data (2019-20) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2019-20.xlsx
[8/17] Enrollment Data (2018-19) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2018-19.xlsx
[9/17] Enrollment Data (2017-18) (Excel)
  -> downloaded: data\raw\hawaii\enrollment\DOEenrollment2017

In [14]:
# --- Count Hawaii files by category after Phase 3 ---
hi_files = [p for p in (DATA_RAW / "hawaii").rglob("*") if p.is_file()]

hi_counts = {}
for path in hi_files:
    category = path.parent.name
    hi_counts[category] = hi_counts.get(category, 0) + 1

hi_summary = pd.DataFrame(
    [{"category": k, "file_count": v} for k, v in sorted(hi_counts.items())]
)

print(f"Total Hawaii files in data/raw/hawaii/: {len(hi_files)}")
hi_summary

Total Hawaii files in data/raw/hawaii/: 17


,category,file_count
0,enrollment,9
1,other,8


---
## Phase 3.4 — Expand Hawaii seed pages (more HIDOE links)

Phase 3.2 only scanned **2 pages**. HIDOE spreads files across other report pages too.

This cell **adds more official seed URLs**, re-scans, and downloads any **new** links not already in `hi_discovered_links.csv`.

In [15]:
# --- Extra official HIDOE pages with additional download links ---
EXTRA_HARVEST_SEED_PAGES = [
    {
        "state": "hawaii",
        "label": "Research and Data Requests (forms + public reports index PDF)",
        "url": "https://hawaiipublicschools.org/data-reports/research-and-data-requests/",
    },
    {
        "state": "hawaii",
        "label": "Find Your School (school list Excel)",
        "url": "https://hawaiipublicschools.org/enrolling-in-school/find-your-school/",
    },
    {
        "state": "hawaii",
        "label": "School Reports (Hawaiian language page — sometimes has extra links)",
        "url": "https://hawaiipublicschools.org/haw/data-reports/school-reports/",
    },
]

# Load URLs we already discovered so we only download NEW links
already_known = set()
if HI_DISCOVERED_CSV.exists():
    already_known = set(pd.read_csv(HI_DISCOVERED_CSV)["url"])

extra_rows = []

for seed in EXTRA_HARVEST_SEED_PAGES:
    page_url = seed["url"]
    print(f"Scanning: {seed['label']}")

    try:
        html = fetch_page_html(page_url)
        links = extract_file_links(page_url, html, seed["state"])
        new_links = [row for row in links if row["url"] not in already_known]
        print(f"  Found {len(links)} total, {len(new_links)} new\n")
        extra_rows.extend(new_links)
        already_known.update(row["url"] for row in links)

    except requests.HTTPError as err:
        print(f"  SKIPPED: {err}\n")

extra_hi_df = pd.DataFrame(extra_rows)

if not extra_hi_df.empty:
    extra_hi_df = extra_hi_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

    # Append to saved discovery file
    combined_hi = pd.concat([pd.read_csv(HI_DISCOVERED_CSV), extra_hi_df], ignore_index=True)
    combined_hi = combined_hi.drop_duplicates(subset=["url"]).reset_index(drop=True)
    combined_hi.to_csv(HI_DISCOVERED_CSV, index=False)

    print(f"New links to download: {len(extra_hi_df)}")
    print(f"Total Hawaii links in catalog: {len(combined_hi)}")
    extra_hi_df
else:
    print("No new links found on extra pages.")

Scanning: Research and Data Requests (forms + public reports index PDF)
  Found 16 total, 15 new

Scanning: Find Your School (school list Excel)
  Found 3 total, 2 new

Scanning: School Reports (Hawaiian language page — sometimes has extra links)
  Found 5 total, 0 new

New links to download: 13
Total Hawaii links in catalog: 30


In [17]:
# Download only the NEW links from Phase 3.4
if "extra_hi_df" in globals() and not extra_hi_df.empty:
    for i, row in extra_hi_df.iterrows():
        print(f"[{i + 1}/{len(extra_hi_df)}] {row['link_text'][:70]}")
        result = download_file(
            url=row["url"],
            state=row["state"],
            category=row["category"],
            description=row["description"],
        )
        print(f"  -> {result['status']}: {result['local_path']}")
        time.sleep(SECONDS_BETWEEN_DOWNLOADS)

    rebuild_manifest_from_log()
else:
    print("Nothing new to download.")

# Always show the current Hawaii file count (whether or not we downloaded new files)
hi_files = [p for p in (DATA_RAW / "hawaii").rglob("*") if p.is_file()]
print(f"\nTotal Hawaii files now: {len(hi_files)}")

[1/13] Overview of HIDOE Research Application Process (PDF)
  -> downloaded: data\raw\hawaii\other\HIDOEResearchApplicationProcessOverview.pdf
[2/13] Publicly Available HIDOE Reports (PDF)
  -> downloaded: data\raw\hawaii\other\Publicly-Available-HIDOE-Reports.pdf
[3/13] Research Coursework Form (PDF)
  -> downloaded: data\raw\hawaii\other\ResearchCoursework.pdf
[4/13] Data Request Form (Excel)
  -> downloaded: data\raw\hawaii\other\DataRequestForm-rev032026.xlsx
[5/13] View the YRBS Data Request Form (Excel)
  -> downloaded: data\raw\hawaii\other\YRBSDataRequestForm.xlsx
[6/13] BOE Policy 500-21 Student Information and Confidential Records (PDF)
  -> downloaded: data\raw\hawaii\other\Student Information and Confidential Records.pdf
[7/13] Application for Proposed Research Projects (PDF)
  -> downloaded: data\raw\hawaii\other\ResearchApplication.pdf
[8/13] Research Review Committee Meeting Schedule 2025-26 (PDF)
  -> downloaded: data\raw\hawaii\other\Research-Schedule.pdf
[9/13] Adult 

---
# Phase 4 — Federal sources (NCES + CRDC)

**Goal:** Download **U.S. Department of Education** public files that include **both Hawaii and Virginia**.

**Why this matters for Hawaii:** The state HIDOE site has ~30 direct files. Federal sources add **discipline, enrollment, staffing, school directories**, etc. for **all states** — you filter to HI/VA later if needed.

**Two federal sources in this phase:**

| Source | What it provides | How we get URLs |
|--------|------------------|-----------------|
| **NCES CCD** | School & state enrollment/staff directories | BeautifulSoup on NCES catalog pages |
| **CRDC** (via data.ed.gov) | Discipline, enrollment, teachers, AP, etc. | API search (like Virginia Phase 2) |

Files save under **`data/raw/federal/{category}/`** (not hawaii/ or virginia/ — national files apply to both states).

## Phase 4.1 — Federal download helper

Same idea as `download_file()`, but saves to **`data/raw/federal/{category}/`**.

We log `state = "federal"` in the manifest so you can tell national files apart from HI/VA state downloads.

In [18]:
FEDERAL_RAW = DATA_RAW / "federal"
FEDERAL_DISCOVERED_CSV = LOGS_DIR / "federal_discovered_links.csv"

# Create federal category folders (same categories as HI/VA)
for category in CATEGORIES:
    (FEDERAL_RAW / category).mkdir(parents=True, exist_ok=True)


def download_federal_file(
    url: str,
    category: str,
    description: str,
    *,
    skip_if_exists: bool = True,
    timeout_seconds: int = 120,
) -> dict:
    """
    Download one federal public file into data/raw/federal/{category}/.

    Works like download_file(), but:
      - state is always logged as 'federal'
      - files apply to ALL states (including Hawaii and Virginia)
    """
    if category not in CATEGORIES:
        raise ValueError(f"category must be one of {CATEGORIES}, got {category!r}")

    dest_dir = FEDERAL_RAW / category
    dest_dir.mkdir(parents=True, exist_ok=True)

    with requests.get(
        url,
        headers=DEFAULT_HEADERS,
        timeout=timeout_seconds,
        stream=True,
    ) as response:
        response.raise_for_status()
        filename = _filename_from_url(url, response)
        dest_path = dest_dir / filename

        if skip_if_exists and dest_path.exists():
            record = {
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "status": "skipped_exists",
                "state": "federal",
                "category": category,
                "description": description,
                "url": url,
                "local_path": str(dest_path.relative_to(PROJECT_ROOT)),
                "bytes": dest_path.stat().st_size,
            }
            with DOWNLOAD_LOG.open("a", encoding="utf-8") as log_file:
                log_file.write(json.dumps(record) + "\n")
            return record

        bytes_written = 0
        with dest_path.open("wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
                    bytes_written += len(chunk)

    record = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "status": "downloaded",
        "state": "federal",
        "category": category,
        "description": description,
        "url": url,
        "local_path": str(dest_path.relative_to(PROJECT_ROOT)),
        "bytes": bytes_written,
    }
    with DOWNLOAD_LOG.open("a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(record) + "\n")
    return record


print("Federal download helper ready.")

Federal download helper ready.


## Phase 4.2 — NCES Common Core of Data (CCD)

**NCES** publishes national school & state education statistics.

We scrape the official NCES catalog page for `.zip` links, pick the **most recent school-universe CSV zip** available, and download it. Inside the zip are **all U.S. states** — including Hawaii (`ST=15`) and Virginia (`ST=51`).

In [19]:
NCES_SCHOOL_CATALOG = "https://nces.ed.gov/ccd/pubschuniv.asp"
NCES_STATE_CATALOG = "https://nces.ed.gov/ccd/stnfis.asp"


def harvest_nces_zip_links(catalog_url: str) -> list[str]:
    """Return all .zip links found on an NCES CCD catalog page."""
    html = requests.get(catalog_url, headers=HTML_HEADERS, timeout=60).text
    soup = BeautifulSoup(html, "html.parser")

    links = []
    for tag in soup.find_all("a", href=True):
        full_url = urljoin(catalog_url, tag["href"])
        if full_url.lower().endswith(".zip"):
            links.append(full_url)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(links))


def pick_newest_ccd_school_csv_zip(links: list[str]) -> str | None:
    """
    From many NCES zip URLs, pick the newest 'ccd_sch' CSV file.

    Example filename: ccd_sch_029_1819_w_0a_04082019_csv.zip
    We sort by the 4-digit school-year code embedded in the name (1819 = 2018-19).
    """
    candidates = [
        u for u in links
        if "ccd_sch_029" in u.lower() and "csv" in u.lower()
    ]
    if not candidates:
        return None

    def year_key(url: str) -> int:
        # Find a 4-digit chunk like 1819, 1718, 1516 in the filename
        match = re.search(r"_(\d{4})_", url)
        return int(match.group(1)) if match else 0

    return max(candidates, key=year_key)


# --- Discover NCES links ---
nces_school_links = harvest_nces_zip_links(NCES_SCHOOL_CATALOG)
nces_state_links = harvest_nces_zip_links(NCES_STATE_CATALOG)

best_school_zip = pick_newest_ccd_school_csv_zip(nces_school_links)
print(f"NCES school catalog links found: {len(nces_school_links)}")
print(f"Newest school-universe CSV zip: {best_school_zip}")

# Also grab a state nonfiscal Excel zip (state-level enrollment/staff counts)
state_xls_candidates = [
    u for u in nces_state_links
    if u.lower().endswith(".zip") and "xls" in u.lower() and "documentation" not in u.lower()
]

def state_xls_key(url: str) -> int:
    match = re.search(r"st(\d{2,3})", url.lower())
    return int(match.group(1)) if match else 0

best_state_zip = max(state_xls_candidates, key=state_xls_key) if state_xls_candidates else None
print(f"Newest state nonfiscal xls zip: {best_state_zip}")

# Build a small list of NCES files to download
NCES_DOWNLOADS = []

if best_school_zip:
    NCES_DOWNLOADS.append(
        {
            "url": best_school_zip,
            "category": "enrollment",
            "description": "NCES CCD Public School Universe (national CSV zip; filter ST=15 HI, ST=51 VA)",
        }
    )

if best_state_zip:
    NCES_DOWNLOADS.append(
        {
            "url": best_state_zip,
            "category": "enrollment",
            "description": "NCES CCD State Nonfiscal Survey (national; state-level enrollment/staff)",
        }
    )

pd.DataFrame(NCES_DOWNLOADS)

NCES school catalog links found: 229
Newest school-universe CSV zip: https://nces.ed.gov/ccd/Data/zip/ccd_sch_029_1819_w_0a_04082019_csv.zip
Newest state nonfiscal xls zip: https://nces.ed.gov/ccd/data/zip/st991bxls.zip


,url,category,description
0,https://nces.ed.gov/ccd/Data/zip/ccd_sch_029_1...,enrollment,NCES CCD Public School Universe (national CSV ...
1,https://nces.ed.gov/ccd/data/zip/st991bxls.zip,enrollment,NCES CCD State Nonfiscal Survey (national; sta...


In [20]:
# Download NCES files discovered above
for item in NCES_DOWNLOADS:
    print(f"Downloading: {item['description']}")
    result = download_federal_file(
        url=item["url"],
        category=item["category"],
        description=item["description"],
    )
    print(f"  -> {result['status']}: {result['local_path']} ({result['bytes']:,} bytes)\n")
    time.sleep(SECONDS_BETWEEN_DOWNLOADS)

Downloading: NCES CCD Public School Universe (national CSV zip; filter ST=15 HI, ST=51 VA)
  -> downloaded: data\raw\federal\enrollment\ccd_sch_029_1819_w_0a_04082019_csv.zip (5,116,009 bytes)

Downloading: NCES CCD State Nonfiscal Survey (national; state-level enrollment/staff)
  -> downloaded: data\raw\federal\enrollment\st991bxls.zip (77,324 bytes)



## Phase 4.3 — CRDC via data.ed.gov API

**CRDC** = Civil Rights Data Collection (discipline, enrollment, AP, teachers, etc.).

The main CRDC website uses JavaScript, but **data.ed.gov** lists **1,000+ direct `.xlsx`/`.csv` download links**.

We query the API (same CKAN style as Virginia Phase 2), keep **recent collection years**, then download each file.

In [21]:
ED_GOV_CKAN_API = "https://data.ed.gov/api/3/action"
CRDC_SEARCH_QUERY = "civil rights data collection"

# Only download files from these CRDC collection folders (recent + useful)
CRDC_YEAR_TAGS = ("2017-2018", "2017-18", "2020-2021", "2020-21", "2021-2022", "2021-22")

CRDC_ALLOWED_FORMATS = {"CSV", "XLSX", "XLS", "ZIP"}


def edgov_package_search(query: str, rows: int = 100, start: int = 0) -> dict:
    """Search datasets on data.ed.gov (CKAN API — same idea as Virginia's API)."""
    url = f"{ED_GOV_CKAN_API}/package_search"
    response = requests.get(
        url,
        params={"q": query, "rows": rows, "start": start},
        headers=DEFAULT_HEADERS,
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    if not payload.get("success"):
        raise RuntimeError(payload)
    return payload["result"]


# --- Page through all CRDC packages on data.ed.gov ---
crdc_rows = []
start = 0

while True:
    page = edgov_package_search(CRDC_SEARCH_QUERY, rows=100, start=start)
    batch = page["results"]

    for package in batch:
        title = package.get("title") or ""
        for resource in package.get("resources", []):
            fmt = (resource.get("format") or "").upper().strip()
            url = (resource.get("url") or "").strip()

            if not url or fmt not in CRDC_ALLOWED_FORMATS:
                continue

            # Keep only recent CRDC cycles (reduces 1000+ files to ~100 manageable ones)
            if not any(tag in url or tag in title for tag in CRDC_YEAR_TAGS):
                continue

            category = guess_category(title + " " + url)

            crdc_rows.append(
                {
                    "source": "CRDC",
                    "category": category,
                    "dataset_title": title,
                    "format": fmt,
                    "url": url,
                    "description": f"CRDC: {title} — {resource.get('name') or fmt}",
                }
            )

    print(f"Fetched CRDC packages {start + 1}–{start + len(batch)} of {page['count']}")
    start += 100
    if start >= page["count"]:
        break

crdc_df = pd.DataFrame(crdc_rows)

if not crdc_df.empty:
    before = len(crdc_df)
    crdc_df = crdc_df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    print(f"\nCRDC file links to download: {len(crdc_df)} (removed {before - len(crdc_df)} duplicates)")
else:
    print("No CRDC links found.")

crdc_df.head(10)

Fetched CRDC packages 1–100 of 135
Fetched CRDC packages 101–135 of 135

CRDC file links to download: 118 (removed 4 duplicates)


,source,category,dataset_title,format,url,description
0,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
1,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
2,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
3,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
4,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
5,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
6,CRDC,discipline,2017-18 Arrests Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Arrests Civil Rights Data Collec...
7,CRDC,discipline,2017-18 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Discipline Civil Rights Data Col...
8,CRDC,discipline,2017-18 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Discipline Civil Rights Data Col...
9,CRDC,discipline,2017-18 Discipline Civil Rights Data Collection,XLSX,https://civilrightsdata.ed.gov/assets/download...,CRDC: 2017-18 Discipline Civil Rights Data Col...


In [22]:
# Download all CRDC files (may take several minutes)
crdc_failures = []

for i, row in crdc_df.iterrows():
    print(f"[{i + 1}/{len(crdc_df)}] {row['description'][:75]}...")
    try:
        result = download_federal_file(
            url=row["url"],
            category=row["category"],
            description=row["description"],
        )
        print(f"  -> {result['status']}: {Path(result['local_path']).name}")
    except requests.HTTPError as err:
        crdc_failures.append({"url": row["url"], "error": str(err)})
        print(f"  -> FAILED: {err}")

    time.sleep(SECONDS_BETWEEN_DOWNLOADS)

print(f"\nCRDC failures: {len(crdc_failures)}")

[1/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — Offenses...
  -> downloaded: Offenses.xlsx
[2/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — School-Related Arrests...
  -> downloaded: School-Related-Arrest_by-no-disability.xlsx
[3/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — School-Related Arrests...
  -> downloaded: School-Related-Arrest_by-disability-and-no.xlsx
[4/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — School-Related Arrests...
  -> downloaded: School-Related-Arrest_by-disability.xlsx
[5/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — Referred to Law Enforc...
  -> downloaded: Referred-to-Law-Enforcement_by-no-disability.xlsx
[6/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — Referred to Law Enforc...
  -> downloaded: Referred-to-Law-Enforcement_by-disability-and-no.xlsx
[7/118] CRDC: 2017-18 Arrests Civil Rights Data Collection — Referred to Law Enforc...
  -> downloaded: Referred-to-Law-Enforcement_by-disa

In [23]:
# Save combined federal discovery table (NCES + CRDC)
federal_catalog_parts = []

if NCES_DOWNLOADS:
    federal_catalog_parts.append(pd.DataFrame(NCES_DOWNLOADS))

if not crdc_df.empty:
    federal_catalog_parts.append(crdc_df)

if federal_catalog_parts:
    federal_catalog = pd.concat(federal_catalog_parts, ignore_index=True)
    federal_catalog.to_csv(FEDERAL_DISCOVERED_CSV, index=False)
    print(f"Saved federal catalog -> {FEDERAL_DISCOVERED_CSV.relative_to(PROJECT_ROOT)}")

manifest_df = rebuild_manifest_from_log()

# Summary: federal files by category
fed_files = [p for p in FEDERAL_RAW.rglob("*") if p.is_file()]
fed_counts = {}
for path in fed_files:
    cat = path.parent.name
    fed_counts[cat] = fed_counts.get(cat, 0) + 1

print(f"Total federal files: {len(fed_files)}")
pd.DataFrame([{"category": k, "file_count": v} for k, v in sorted(fed_counts.items())])

Saved federal catalog -> logs\federal_discovered_links.csv
Total federal files: 104


,category,file_count
0,discipline,28
1,enrollment,27
2,other,43
3,teachers,2
4,test_scores,4


---
# Phase 5 — Parse HIDOE catalog PDF → discover & download new links

**Goal:** Read `Publicly-Available-HIDOE-Reports.pdf` (already in `data/raw/hawaii/other/`), extract official URLs, scan those pages for file links, and download anything we do **not** already have.

**Why this helps Hawaii:** The PDF is HIDOE’s own list of public reports. It points to sources we did not scan in Phase 3 — especially **`https://hcnp.hawaii.gov/fiscal/`** (dozens of Excel/PDF files on free/reduced lunch and related fiscal data).

**Note:** Many ARCH links in the PDF are **dashboard pages** (JavaScript apps) with no direct `.xlsx` links. We still try them; if zero files appear, that is expected — document it later.

## Phase 5.1 — Install PDF reader and set paths

We use **`pdfplumber`** to read text out of the PDF (like copy-paste, but in Python).

In [24]:
# Run once if pdfplumber is not installed yet
%pip install -q pdfplumber

import pdfplumber

# The catalog PDF we downloaded in Phase 3.4
HIDOE_CATALOG_PDF = DATA_RAW / "hawaii" / "other" / "Publicly-Available-HIDOE-Reports.pdf"
PHASE5_DISCOVERED_CSV = LOGS_DIR / "hi_phase5_discovered_links.csv"

print("Catalog PDF exists:", HIDOE_CATALOG_PDF.exists())
print("Path:", HIDOE_CATALOG_PDF)

Note: you may need to restart the kernel to use updated packages.
Catalog PDF exists: True
Path: c:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\hawaii\other\Publicly-Available-HIDOE-Reports.pdf



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\saihaj\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


## Phase 5.2 — Extract URLs from the PDF and harvest file links

**Step A:** Pull all `http://...` strings from the PDF text.

**Step B:** Fix URLs that were **split across lines** in the PDF (common with PDF text extraction).

**Step C:** Visit each seed URL — if it is a webpage, use BeautifulSoup (same as Phase 3) to find `.xlsx` / `.pdf` links.

In [25]:
def normalize_pdf_url(raw_url: str) -> str:
    """
    Clean a URL extracted from PDF text.

    PDF line breaks sometimes truncate URLs or leave trailing punctuation.
    """
    url = raw_url.strip().rstrip(".,;")

    # Fix known broken URLs from this specific HIDOE catalog PDF
    if url.endswith("/reports/trend-"):
        url = url + "report"
    if url.rstrip("/").endswith("/reports/strive"):
        url = url.rstrip("/") + "-hi-performance"

    return url


def extract_urls_from_pdf(pdf_path: Path) -> tuple[list[str], str]:
    """
    Read the PDF and return (list of URLs, full text).

    Uses regex to find http/https links embedded in the document.
    """
    full_text = ""

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # extract_text() turns PDF layout into plain strings
            full_text += (page.extract_text() or "") + "\n"

    # Find http/https links in the text
    found = re.findall(r"https?://[^\s\)\],>]+", full_text)

    # The PDF also mentions paths split across lines; add known ARCH paths manually
    manual_arch_seeds = [
        "http://arch.k12.hi.us/reports/hidoe-data-book",
    ]

    urls = [normalize_pdf_url(u) for u in found + manual_arch_seeds]

    # Keep unique URLs in stable order
    urls = list(dict.fromkeys(urls))

    return urls, full_text


def harvest_files_from_seed_urls(seed_urls: list[str], state: str = "hawaii") -> list[dict]:
    """
    For each seed URL from the PDF:
      - if it already points to a file -> keep it
      - if it is a webpage -> scrape file links with BeautifulSoup
    """
    rows = []

    for seed_url in seed_urls:
        path_lower = urlparse(seed_url).path.lower()

        # Case 1: seed URL is already a direct file link
        if path_lower.endswith(FILE_EXTENSIONS):
            rows.append(
                {
                    "state": state,
                    "category": guess_category(seed_url),
                    "link_text": Path(path_lower).name,
                    "url": seed_url,
                    "source_page": "HIDOE catalog PDF (direct file URL)",
                    "description": f"From HIDOE catalog PDF: {seed_url}",
                }
            )
            continue

        # Case 2: seed URL is a webpage — scrape for downloads
        print(f"Scanning seed page: {seed_url}")
        try:
            html = fetch_page_html(seed_url)
            page_links = extract_file_links(seed_url, html, state)
            print(f"  -> found {len(page_links)} file link(s)")
            rows.extend(page_links)
        except requests.HTTPError as err:
            print(f"  -> SKIPPED (HTTP error): {err}")

    return rows


# --- Run extraction ---
pdf_seed_urls, pdf_full_text = extract_urls_from_pdf(HIDOE_CATALOG_PDF)

print(f"URLs extracted from PDF: {len(pdf_seed_urls)}")
for u in pdf_seed_urls:
    print(" ", u)

# Harvest downloadable files from those seed pages
phase5_rows = harvest_files_from_seed_urls(pdf_seed_urls)
phase5_df = pd.DataFrame(phase5_rows)

if not phase5_df.empty:
    before = len(phase5_df)
    phase5_df = phase5_df.drop_duplicates(subset=["url"]).reset_index(drop=True)

    # Skip URLs we already collected in Phase 3 / 3.4
    if HI_DISCOVERED_CSV.exists():
        already_have = set(pd.read_csv(HI_DISCOVERED_CSV)["url"])
        phase5_df = phase5_df[~phase5_df["url"].isin(already_have)].reset_index(drop=True)

    print(f"\nNew Hawaii links from Phase 5: {len(phase5_df)} (removed {before - len(phase5_df)} duplicates/already-known)")
    phase5_df.to_csv(PHASE5_DISCOVERED_CSV, index=False)
    print(f"Saved -> {PHASE5_DISCOVERED_CSV.relative_to(PROJECT_ROOT)}")
else:
    print("\nNo new file links found from PDF seeds.")

phase5_df.head(10)

URLs extracted from PDF: 9
  http://arch.k12.hi.us/reports/ssir
  http://arch.k12.hi.us/reports/sqs
  http://arch.k12.hi.us/reports/trend-report
  http://arch.k12.hi.us/reports/strive-hi-performance
  http://arch.k12.hi.us/reports/essa
  http://arch.k12.hi.us/reports/hidoe
  https://hidoedata.org/
  https://hcnp.hawaii.gov/fiscal/
  http://arch.k12.hi.us/reports/hidoe-data-book
Scanning seed page: http://arch.k12.hi.us/reports/ssir
  -> found 0 file link(s)
Scanning seed page: http://arch.k12.hi.us/reports/sqs
  -> found 0 file link(s)
Scanning seed page: http://arch.k12.hi.us/reports/trend-report
  -> found 0 file link(s)
Scanning seed page: http://arch.k12.hi.us/reports/strive-hi-performance
  -> found 0 file link(s)
Scanning seed page: http://arch.k12.hi.us/reports/essa
  -> found 0 file link(s)
Scanning seed page: http://arch.k12.hi.us/reports/hidoe
  -> found 0 file link(s)
Scanning seed page: https://hidoedata.org/
  -> found 0 file link(s)
Scanning seed page: https://hcnp.hawaii

,state,category,link_text,url,source_page,description
0,hawaii,other,FY 12-13 A-133 Audit Certification,https://hcnp.hawaii.gov/wp-content/uploads/201...,https://hcnp.hawaii.gov/fiscal/,FY 12-13 A-133 Audit Certification (from https...
1,hawaii,other,ICN Financial Management Information System Ma...,https://hcnp.hawaii.gov/wp-content/uploads/202...,https://hcnp.hawaii.gov/fiscal/,ICN Financial Management Information System Ma...
2,hawaii,other,ICN Procurement in the 21st Century Resource M...,https://hcnp.hawaii.gov/wp-content/uploads/202...,https://hcnp.hawaii.gov/fiscal/,ICN Procurement in the 21st Century Resource M...
3,hawaii,other,SY 12-13 NSLP Annual Financial Report of Child...,https://hcnp.hawaii.gov/wp-content/uploads/201...,https://hcnp.hawaii.gov/fiscal/,SY 12-13 NSLP Annual Financial Report of Child...
4,hawaii,other,CFDA Numbers,https://hcnp.hawaii.gov/wp-content/uploads/201...,https://hcnp.hawaii.gov/fiscal/,CFDA Numbers (from https://hcnp.hawaii.gov/fis...
5,hawaii,other,DUNS-Certification Form,https://hcnp.hawaii.gov/wp-content/uploads/201...,https://hcnp.hawaii.gov/fiscal/,DUNS-Certification Form (from https://hcnp.haw...
6,hawaii,other,NOTICE: SY 2021-2022 Schedule for Submitting O...,https://hcnp.hawaii.gov/wp-content/uploads/202...,https://hcnp.hawaii.gov/fiscal/,NOTICE: SY 2021-2022 Schedule for Submitting O...
7,hawaii,other,NOTICE: SY 2022-2023 Schedule for Submitting O...,https://hcnp.hawaii.gov/wp-content/uploads/202...,https://hcnp.hawaii.gov/fiscal/,NOTICE: SY 2022-2023 Schedule for Submitting O...
8,hawaii,other,NOTICE: SY 2023-2024 Schedule for Submitting O...,https://hcnp.hawaii.gov/wp-content/uploads/202...,https://hcnp.hawaii.gov/fiscal/,NOTICE: SY 2023-2024 Schedule for Submitting O...
9,hawaii,other,NOTICE: SY 2024-2025 Schedule for Submitting O...,https://hcnp.hawaii.gov/wp-content/uploads/202...,https://hcnp.hawaii.gov/fiscal/,NOTICE: SY 2024-2025 Schedule for Submitting O...


## Phase 5.3 — Download new Hawaii files

Uses the same **`download_file()`** helper as earlier phases.
Files go to `data/raw/hawaii/{category}/`.

In [26]:
phase5_failures = []

if phase5_df.empty:
    print("Nothing new to download in Phase 5.")
else:
    for i, row in phase5_df.iterrows():
        print(f"[{i + 1}/{len(phase5_df)}] {row['link_text'][:70]}")

        try:
            result = download_file(
                url=row["url"],
                state=row["state"],
                category=row["category"],
                description=row["description"],
            )
            print(f"  -> {result['status']}: {result['local_path']}")

        except requests.HTTPError as err:
            phase5_failures.append({"url": row["url"], "error": str(err)})
            print(f"  -> FAILED: {err}")

        time.sleep(SECONDS_BETWEEN_DOWNLOADS)

    # Append new URLs to the master Hawaii discovery catalog
    if HI_DISCOVERED_CSV.exists():
        combined_hi = pd.concat(
            [pd.read_csv(HI_DISCOVERED_CSV), phase5_df],
            ignore_index=True,
        )
    else:
        combined_hi = phase5_df.copy()

    combined_hi = combined_hi.drop_duplicates(subset=["url"]).reset_index(drop=True)
    combined_hi.to_csv(HI_DISCOVERED_CSV, index=False)

    rebuild_manifest_from_log()

    print(f"\nPhase 5 failures: {len(phase5_failures)}")

[1/52] FY 12-13 A-133 Audit Certification
  -> downloaded: data\raw\hawaii\other\FY-12-13-A-133-Audit-Certification.xls
[2/52] ICN Financial Management Information System Manual
  -> downloaded: data\raw\hawaii\other\ICN-Financial-Management-Information-System-2nd-Edition-2018.pdf
[3/52] ICN Procurement in the 21st Century Resource Manual
  -> downloaded: data\raw\hawaii\other\ICN-Procurement-in-the-21st-Century-Resource-Manual-2015.pdf
[4/52] SY 12-13 NSLP Annual Financial Report of Child Nutrition Programs
  -> downloaded: data\raw\hawaii\other\HCNP-SY2012-2013-NSLP-ANNUAL-FINANCIAL-REPORT-OF-Child-Nutrition-Programs-Due-Oct-15.xls
[5/52] CFDA Numbers
  -> downloaded: data\raw\hawaii\other\CFDA-Numbers.pdf
[6/52] DUNS-Certification Form
  -> downloaded: data\raw\hawaii\other\FISCAL-DUNS-Certification-Form.xls
[7/52] NOTICE: SY 2021-2022 Schedule for Submitting Online Claims
  -> downloaded: data\raw\hawaii\other\Notice-SY-21-22-Schedule-for-Submitting-Online-Claims.pdf
[8/52] NOTICE:

In [27]:
# --- Phase 5 summary ---
hi_files = [p for p in (DATA_RAW / "hawaii").rglob("*") if p.is_file()]

hi_counts = {}
for path in hi_files:
    category = path.parent.name
    hi_counts[category] = hi_counts.get(category, 0) + 1

print(f"Total Hawaii files now: {len(hi_files)}")
pd.DataFrame([{"category": k, "file_count": v} for k, v in sorted(hi_counts.items())])

# Reminder about ARCH dashboard links (no direct downloads found)
arch_seeds = [u for u in pdf_seed_urls if "arch.k12.hi.us" in u]
if arch_seeds:
    print("\nARCH dashboard seeds from PDF (usually no direct file links):")
    for u in arch_seeds:
        print(" ", u)

Total Hawaii files now: 78

ARCH dashboard seeds from PDF (usually no direct file links):
  http://arch.k12.hi.us/reports/ssir
  http://arch.k12.hi.us/reports/sqs
  http://arch.k12.hi.us/reports/trend-report
  http://arch.k12.hi.us/reports/strive-hi-performance
  http://arch.k12.hi.us/reports/essa
  http://arch.k12.hi.us/reports/hidoe
  http://arch.k12.hi.us/reports/hidoe-data-book


---
# Phase 6 — Better categories + Hawaii data from federal files

**Problem:** Many Hawaii folders show **0** because (1) keyword sorting was too weak (`hcnp` fiscal files landed in `other/`), and (2) test/discipline/teacher data lives in **federal** files we already downloaded.

**This phase does three things:**

1. **Improved keyword rules** → re-sort existing Hawaii files into the right folders.
2. **NCES school directory** → extract **Hawaii-only** rows from the national zip (`ST = HI`).
3. **CRDC 2017–18 zip** → extract **Hawaii-only** CSVs (discipline, AP, enrollment, teachers, etc.) into `data/cleaned/hawaii/`.

**Important:** `data/raw/` stays your proof of originals. We **move** Hawaii state files between subfolders (same bytes). Federal extracts go to **`data/cleaned/hawaii/`** (copies filtered to HI).

## Phase 6.1 — Improved keyword matching

We replace the simple `guess_category()` rules with **`guess_category_improved()`** that knows about:

- **hcnp.hawaii.gov** fiscal / free-reduced lunch files
- **HIDOE enrollment** file names
- **Title I**, **research forms**, etc.

Then we **move** Hawaii files from the wrong folder to the correct one (using the download log for URL/description).

In [28]:
import shutil

# State abbreviations used inside federal CSV files
HI_ABBR = "HI"
VA_ABBR = "VA"

DATA_CLEANED_HI = DATA_CLEANED / "hawaii"
DATA_CLEANED_VA = DATA_CLEANED / "virginia"
for category in CATEGORIES:
    (DATA_CLEANED_HI / category).mkdir(parents=True, exist_ok=True)
    (DATA_CLEANED_VA / category).mkdir(parents=True, exist_ok=True)

RECATEGORIZE_LOG = LOGS_DIR / "hi_recategorize_log.jsonl"


def guess_category_improved(text: str) -> str:
    """
    Improved folder guess using URL, filename, and description text.

    Order matters: first match wins (most specific rules first).
    """
    t = text.lower()

    rules = [
        ("enrollment", [
            "hidoeenrollment", "doeenrollment", "enrollment", "membership",
            "fall membership", "student count", "list-of-schools",
            "free-reduced", "fr-pct", "sfa", "october data", "eligible students",
            "all-hi-sfas", "doe-only", "hi-schools",
        ]),
        ("financials", [
            "hcnp.hawaii.gov", "fiscal", "budget", "expenditure", "per pupil",
            "audit", "nslp", "cfda", "a-133", "duns", "procurement", "financial",
            "weighted student formula", "operating budget",
        ]),
        ("test_scores", [
            "sol", "assessment", "test score", "pass rate", "strive", "naep",
            "advanced placement", "algebra", "sat", "act", "accreditation",
        ]),
        ("teachers", [
            "teacher", "staff", "personnel", "licens", "qualification",
            "cssp", "employment report", "newly employed",
        ]),
        ("discipline", [
            "discipline", "suspension", "expulsion", "absenteeism", "climate",
            "offenses", "harassment", "bullying", "wellness", "saws",
        ]),
    ]

    for category, keywords in rules:
        if any(word in t for word in keywords):
            return category

    return "other"


# Override the old function name so later cells use the improved rules
guess_category = guess_category_improved

print("guess_category_improved() is ready.")

guess_category_improved() is ready.


In [29]:
# --- Re-sort existing Hawaii files in data/raw/hawaii/ ---

# Build a lookup: local filename -> metadata from the download log
manifest = pd.read_csv(MANIFEST_CSV)
hi_manifest = manifest[manifest["state"] == "hawaii"].copy()

# Keep the most recent log entry per local path
hi_manifest = hi_manifest.sort_values("timestamp_utc").drop_duplicates(
    subset=["local_path"], keep="last"
)

moves = []

for file_path in sorted((DATA_RAW / "hawaii").rglob("*")):
    if not file_path.is_file():
        continue

    current_category = file_path.parent.name
    rel_path = str(file_path.relative_to(PROJECT_ROOT)).replace("/", "\\")

    # Find metadata for this file (match on normalized path)
    match = hi_manifest[hi_manifest["local_path"].str.replace("/", "\\") == rel_path]

    if not match.empty:
        row = match.iloc[0]
        text_blob = f"{row['url']} {row['description']} {file_path.name}"
    else:
        text_blob = file_path.name

    new_category = guess_category_improved(text_blob)

    if new_category == current_category:
        continue

    dest_path = DATA_RAW / "hawaii" / new_category / file_path.name

    # If same filename already exists in destination, skip (avoid overwrite)
    if dest_path.exists():
        print(f"SKIP (name clash): {file_path.name} -> {new_category}/")
        continue

    dest_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(file_path), str(dest_path))

    record = {
        "file": file_path.name,
        "from_category": current_category,
        "to_category": new_category,
        "new_path": str(dest_path.relative_to(PROJECT_ROOT)),
    }
    moves.append(record)

    with RECATEGORIZE_LOG.open("a", encoding="utf-8") as log_file:
        log_file.write(json.dumps(record) + "\n")

    print(f"MOVED: {file_path.name}  {current_category} -> {new_category}")

print(f"\nTotal files moved: {len(moves)}")

# Show updated Hawaii counts in data/raw/
hi_counts = {}
for path in (DATA_RAW / "hawaii").rglob("*"):
    if path.is_file():
        cat = path.parent.name
        hi_counts[cat] = hi_counts.get(cat, 0) + 1

print("\nHawaii raw files by category (after re-sort):")
pd.DataFrame([{"category": k, "file_count": v} for k, v in sorted(hi_counts.items())])

MOVED: 10-All-HI-SFAs-Alpha.pdf  other -> enrollment
MOVED: 10-All-HI-SFAs-FR-Pct.pdf  other -> enrollment
MOVED: 10-DOE-Only-Alpha.pdf  other -> enrollment
MOVED: 10-DOE-Only-FR-Pct.pdf  other -> enrollment
MOVED: 10-HI-Schools-Alpha.pdf  other -> enrollment
MOVED: 10-HI-Schools-FR-Pct.pdf  other -> enrollment
MOVED: 10-Non-DOE-SFAs-Alpha.pdf  other -> enrollment
MOVED: 10-Non-DOE-SFAs-FR-Pct.pdf  other -> enrollment
MOVED: 11-All-HI-School-Food-Authorities-Alpha.xls  other -> financials
MOVED: 11-All-HIi-School-Food-Authorities-Free-Reduced-Percentage.xls  other -> enrollment
MOVED: 11-DOE-Only-Alpha.xls  other -> enrollment
MOVED: 11-DOE-Only-Free-Reduced-Percentage.xls  other -> enrollment
MOVED: 11-HI-Schools-Alpha.xls  other -> enrollment
MOVED: 11-HI-Schools-Free-Reduced-Percentage.xls  other -> enrollment
MOVED: 11-Non-DOE-School-Food-Authorities-Alpha.xls  other -> financials
MOVED: 11-Non-DOE-School-Food-Authorities-Free-Reduced-Percentage.xls  other -> enrollment
MOVED: 11-O

,category,file_count
0,enrollment,40
1,financials,18
2,other,20


## Phase 6.2 — Extract Hawaii rows from NCES school directory (national zip)

The NCES zip in `data/raw/federal/enrollment/` contains **every U.S. public school**.

We open the CSV inside the zip, keep rows where **`ST == "HI"`**, and save to `data/cleaned/hawaii/enrollment/`.

In [30]:
import zipfile

# Find the NCES school universe zip (downloaded in Phase 4)
nces_zips = list((FEDERAL_RAW / "enrollment").glob("ccd_sch_*csv.zip"))

if not nces_zips:
    print("No NCES school zip found. Run Phase 4 NCES download first.")
else:
    nces_zip_path = nces_zips[0]
    print(f"Using NCES zip: {nces_zip_path.name}")

    with zipfile.ZipFile(nces_zip_path) as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise FileNotFoundError("No CSV inside NCES zip")

        csv_name = csv_names[0]
        print(f"Reading inside zip: {csv_name}")

        with zf.open(csv_name) as f:
            schools = pd.read_csv(f, encoding="latin-1", low_memory=False)

    print(f"National schools rows: {len(schools):,}")

    # ST column = 2-letter state abbreviation (HI, VA, AL, ...)
    hi_schools = schools[schools["ST"] == HI_ABBR].copy()
    va_schools = schools[schools["ST"] == VA_ABBR].copy()

    hi_out = DATA_CLEANED_HI / "enrollment" / "nces_public_schools_HI.csv"
    va_out = DATA_CLEANED_VA / "enrollment" / "nces_public_schools_VA.csv"

    hi_schools.to_csv(hi_out, index=False)
    va_schools.to_csv(va_out, index=False)

    print(f"Hawaii schools saved: {len(hi_schools):,} rows -> {hi_out.relative_to(PROJECT_ROOT)}")
    print(f"Virginia schools saved: {len(va_schools):,} rows -> {va_out.relative_to(PROJECT_ROOT)}")

Using NCES zip: ccd_sch_029_1819_w_0a_04082019_csv.zip
Reading inside zip: ccd_sch_029_1819_w_0a_04082019.csv
National schools rows: 101,923
Hawaii schools saved: 294 rows -> data\cleaned\hawaii\enrollment\nces_public_schools_HI.csv
Virginia schools saved: 2,132 rows -> data\cleaned\virginia\enrollment\nces_public_schools_VA.csv


## Phase 6.3 — Extract Hawaii rows from CRDC 2017–18 public-use zip

The large **`2017-18-crdc-data.zip`** (in `data/raw/federal/other/`) contains **51 school/LEA CSV files** with a column **`LEA_STATE`** (`HI`, `VA`, …).

We filter each CSV to Hawaii only and save copies under `data/cleaned/hawaii/{category}/`.

*(Virginia is included too in the same loop — same project, same zip.)*

In [31]:
def extract_crdc_zip_for_state(zip_path: Path, state_abbr: str, cleaned_root: Path) -> list[dict]:
    """
    Open the CRDC public-use zip and save state-filtered CSVs to cleaned_root.

    Returns a list of metadata dicts (one per file written).
    """
    written = []

    with zipfile.ZipFile(zip_path) as zf:
        csv_inside_zip = [n for n in zf.namelist() if n.lower().endswith(".csv")]

        for inner_csv_path in csv_inside_zip:
            # Example inner path ends with: Advanced Placement.csv
            topic_name = Path(inner_csv_path).stem

            with zf.open(inner_csv_path) as f:
                try:
                    # CRDC CSVs sometimes use Windows/latin-1 encoding
                    df = pd.read_csv(f, low_memory=False, encoding="latin-1")
                except Exception as err:
                    print(f"  SKIP (read error): {topic_name} -> {err}")
                    continue

            # CRDC files use LEA_STATE for the 2-letter state code
            if "LEA_STATE" not in df.columns:
                continue

            state_df = df[df["LEA_STATE"] == state_abbr].copy()
            if state_df.empty:
                continue

            category = guess_category_improved(topic_name)
            safe_name = re.sub(r"[^A-Za-z0-9_-]+", "_", topic_name)[:80]
            out_path = cleaned_root / category / f"crdc_{safe_name}_{state_abbr}.csv"
            state_df.to_csv(out_path, index=False)

            written.append(
                {
                    "state": state_abbr,
                    "category": category,
                    "topic": topic_name,
                    "rows": len(state_df),
                    "local_path": str(out_path.relative_to(PROJECT_ROOT)),
                }
            )

    return written


# Locate the CRDC zip (Phase 4 download)
crdc_zips = list(FEDERAL_RAW.rglob("2017-18-crdc-data.zip"))

if not crdc_zips:
    print("CRDC zip not found. Check data/raw/federal/other/.")
else:
    crdc_zip_path = crdc_zips[0]
    print(f"Using CRDC zip: {crdc_zip_path}")
    print("This may take 2–5 minutes (reading many CSVs inside the zip)...\n")

    hi_crdc = extract_crdc_zip_for_state(crdc_zip_path, HI_ABBR, DATA_CLEANED_HI)
    va_crdc = extract_crdc_zip_for_state(crdc_zip_path, VA_ABBR, DATA_CLEANED_VA)

    print(f"Hawaii CRDC CSV files written: {len(hi_crdc)}")
    print(f"Virginia CRDC CSV files written: {len(va_crdc)}")

    hi_crdc_df = pd.DataFrame(hi_crdc)
    if not hi_crdc_df.empty:
        print(hi_crdc_df.groupby("category").size())

Using CRDC zip: c:\Users\saihaj\Documents\Data Collection - EOQ Lab\data\raw\federal\other\2017-18-crdc-data.zip
This may take 2–5 minutes (reading many CSVs inside the zip)...

Hawaii CRDC CSV files written: 49
Virginia CRDC CSV files written: 49
category
discipline      5
enrollment      2
financials      1
other          35
test_scores     6
dtype: int64


In [32]:
# --- Phase 6 summary: Hawaii data before vs after ---

def count_files(root: Path) -> dict:
    counts = {}
    for path in root.rglob("*"):
        if path.is_file():
            cat = path.parent.name
            counts[cat] = counts.get(cat, 0) + 1
    return counts

print("=== Hawaii STATE files (data/raw/hawaii/) ===")
raw_hi = count_files(DATA_RAW / "hawaii")
print(pd.DataFrame([{"category": k, "files": v} for k, v in sorted(raw_hi.items())]))
print(f"TOTAL raw Hawaii: {sum(raw_hi.values())}")

print("\n=== Hawaii CLEANED files (data/cleaned/hawaii/) — federal HI extracts ===")
clean_hi = count_files(DATA_CLEANED_HI)
print(pd.DataFrame([{"category": k, "files": v} for k, v in sorted(clean_hi.items())]))
print(f"TOTAL cleaned Hawaii: {sum(clean_hi.values())}")

print("\nYou now have:")
print("  • data/raw/hawaii/     = official HIDOE downloads (re-sorted into categories)")
print("  • data/cleaned/hawaii/ = Hawaii-only rows from NCES + CRDC national files")

=== Hawaii STATE files (data/raw/hawaii/) ===
     category  files
0  enrollment     40
1  financials     18
2       other     20
TOTAL raw Hawaii: 78

=== Hawaii CLEANED files (data/cleaned/hawaii/) — federal HI extracts ===
      category  files
0   discipline      5
1   enrollment      3
2   financials      1
3        other     35
4  test_scores      6
TOTAL cleaned Hawaii: 50

You now have:
  • data/raw/hawaii/     = official HIDOE downloads (re-sorted into categories)
  • data/cleaned/hawaii/ = Hawaii-only rows from NCES + CRDC national files


## Phase 6.5 — Combined report summary (for supervisor)

**`data/raw/`** and **`data/cleaned/`** are different layers — not a before/after of the same files.

| Layer | Folder | Source |
|-------|--------|--------|
| **Raw (state)** | `data/raw/{state}/` | Files downloaded directly from Hawaii or Virginia websites |
| **Cleaned (state)** | `data/cleaned/{state}/` | Hawaii- or Virginia-only rows extracted from **federal** NCES + CRDC zips |

In [33]:
# --- Phase 6.5: Combined summary table (paste into supervisor report) ---

def category_file_counts(root: Path) -> dict[str, int]:
    """Count files under root, grouped by parent folder name (category)."""
    counts = {cat: 0 for cat in CATEGORIES}
    if not root.exists():
        return counts
    for path in root.rglob("*"):
        if path.is_file():
            cat = path.parent.name
            counts[cat] = counts.get(cat, 0) + 1
    return counts


def build_combined_summary(raw_root: Path, cleaned_root: Path) -> pd.DataFrame:
    raw_counts = category_file_counts(raw_root)
    clean_counts = category_file_counts(cleaned_root)

    rows = []
    for cat in CATEGORIES:
        r = raw_counts.get(cat, 0)
        c = clean_counts.get(cat, 0)
        rows.append(
            {
                "category": cat,
                "raw_state_downloads": r,
                "cleaned_federal_extracts": c,
                "total_datasets": r + c,
            }
        )

    df = pd.DataFrame(rows)
    totals = pd.DataFrame(
        [
            {
                "category": "TOTAL",
                "raw_state_downloads": df["raw_state_downloads"].sum(),
                "cleaned_federal_extracts": df["cleaned_federal_extracts"].sum(),
                "total_datasets": df["total_datasets"].sum(),
            }
        ]
    )
    return pd.concat([df, totals], ignore_index=True)


STATE_REPORTS = [
    ("Hawaii", DATA_RAW / "hawaii", DATA_CLEANED_HI),
    ("Virginia", DATA_RAW / "virginia", DATA_CLEANED_VA),
]

all_tables = []

for state_name, raw_root, cleaned_root in STATE_REPORTS:
    print("=" * 72)
    print(f"{state_name.upper()} — files by category")
    print(f"  Raw:     {raw_root.relative_to(PROJECT_ROOT)}/")
    print(f"  Cleaned: {cleaned_root.relative_to(PROJECT_ROOT)}/")
    print("=" * 72)

    table = build_combined_summary(raw_root, cleaned_root)
    table.insert(0, "state", state_name)
    print(table.to_string(index=False))
    print()
    all_tables.append(table)

# One combined table for both states (handy for Excel / report appendix)
combined_report = pd.concat(all_tables, ignore_index=True)
print("=" * 72)
print("COMBINED (both states) — copy this block into your report")
print("=" * 72)
print(combined_report.to_string(index=False))

print(
    "\nReminder: raw + cleaned totals count *different sources*."
    "\n  • Raw = state website downloads (original files)"
    "\n  • Cleaned = HI/VA rows cut from federal NCES + CRDC national files"
)

HAWAII — files by category
  Raw:     data\raw\hawaii/
  Cleaned: data\cleaned\hawaii/
 state    category  raw_state_downloads  cleaned_federal_extracts  total_datasets
Hawaii test_scores                    0                         6               6
Hawaii  enrollment                   40                         3              43
Hawaii  financials                   18                         1              19
Hawaii    teachers                    0                         0               0
Hawaii  discipline                    0                         5               5
Hawaii       other                   20                        35              55
Hawaii       TOTAL                   78                        50             128

VIRGINIA — files by category
  Raw:     data\raw\virginia/
  Cleaned: data\cleaned\virginia/
   state    category  raw_state_downloads  cleaned_federal_extracts  total_datasets
Virginia test_scores                   24                         6            

---
## Project complete — next steps

1. Zip `data/raw/` and `data/cleaned/` (or upload to Google Drive).
2. Share `logs/manifest.csv` and this notebook.
3. Fill in `docs/SOURCES.md` using the manifest (state = hawaii | virginia | federal).

**Note for Hawaii test scores:** State dashboards (ARCH) still need Playwright if you want more **HIDOE-published** exports; you already have **CRDC/NCES** school-level data for HI in `data/cleaned/hawaii/`.